# S1 Quantitative Evaluation: INRIA Person Dataset

**Tier 1 quantitative eval** — measure mAP@0.5:0.95 on labelled data (not smoke test).

**Pipeline:**
1. Download INRIA Person dataset (614 positive images, ~50 MB).
2. Parse `.mat` annotations into COCO-format ground truth via `src.inria_loader`.
3. Run each detector on every image, collect `[bbox, score]` per detection.
4. Call `src.eval_mAP.evaluate_map()` to compute mAP via pycocotools.
5. Save per-detector JSON to Google Drive: `MyDrive/hibah-riset-results/s1_quant_inria/`.

**Anti-overclaim**: Only report mAP for detectors in the same tier comparison group.
We run Nano-tier here as the S1 anchor. Larger tier notebooks (03, 04) cover M/L.

In [ ]:
%pip install -q ultralytics pycocotools scipy pillow

import os, sys, json, shutil, time, pathlib
from pathlib import Path

REPO_DIR = Path('/content/hibah-riset')
if not REPO_DIR.exists():
    !git clone --depth=1 https://github.com/feb027/hibah-riset.git $REPO_DIR
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import yaml
with open('configs/s1_quant_inria.yaml') as f:
    cfg = yaml.safe_load(f)['config']
print('Config:', cfg)

In [ ]:
# Mount Google Drive for result persistence
from google.colab import drive
drive.mount('/content/drive')
DRIVE_RESULT_DIR = Path('/content/drive/MyDrive/hibah-riset-results/s1_quant_inria')
DRIVE_RESULT_DIR.mkdir(parents=True, exist_ok=True)
print('Drive result dir:', DRIVE_RESULT_DIR)

## 1. Download INRIA dataset

In [ ]:
!python scripts/download_inria.py --out data/raw/inria 2>&1 | tail -5

from pathlib import Path
INRIA_ROOT = Path('data/raw/inria/inria_person')
candidates = [
    INRIA_ROOT / 'INRIAPerson' / 'Train' / 'pos',
    INRIA_ROOT / 'INRIAPerson' / 'Test' / 'pos',
    INRIA_ROOT / 'Train' / 'pos',
    INRIA_ROOT / 'Test' / 'pos',
]
existing = [c for c in candidates if c.exists()]
print('Found pos dirs:', existing)
if not existing:
    # Try one level deeper
    import subprocess
    sub = subprocess.check_output(['find', str(INRIA_ROOT), '-type', 'd', '-name', 'pos']).decode()
    print('pos dirs via find:\n', sub)

## 2. Parse INRIA annotations → COCO GT
INRIA stores annotations as Matlab `.mat` files. We parse them via `src.inria_loader`.

In [ ]:
from src.inria_loader import inria_to_coco_records

# Discover the actual INRIA root by searching for pos dirs
def find_inria_root(base: Path) -> Path | None:
    for cand in [base, base / 'INRIAPerson', base / 'inria_person']:
        if (cand / 'Train' / 'pos').exists() and (cand / 'Train' / 'posGt').exists():
            return cand
    # One-level deep search
    for p in base.rglob('pos'):
        if (p.parent / 'posGt').exists():
            return p.parent.parent
    return None

actual_root = find_inria_root(INRIA_ROOT)
if actual_root is None:
    raise FileNotFoundError('INRIA pos/posGt dirs not found. Re-run download_inria.py.')
print('Using INRIA root:', actual_root)

gt_records = inria_to_coco_records(actual_root, splits=('Train', 'Test'), category_id=1)
print(f'GT records: {len(gt_records)}')
print(f'Unique images: {len({r["image_id"] for r in gt_records})}')
print('First GT record:', gt_records[0])

## 3. Inference loop — run each detector, collect predictions

In [ ]:
import time
import numpy as np
from PIL import Image
from src.detector import PeopleDetector
from src.eval_mAP import evaluate_map, write_map_result_json

def run_detector_on_inria(detector_alias: str, gt: list[dict], conf_thr: float = 0.001) -> list[dict]:
    """Run `detector_alias` on every INRIA image and return COCO-format predictions."""
    det = PeopleDetector(model_alias=detector_alias, conf=conf_thr)
    preds: list[dict] = []
    t0 = time.time()
    n = len({r['image_id'] for r in gt})
    done = 0
    for img_id in sorted({r['image_id'] for r in gt}):
        rec = next(r for r in gt if r['image_id'] == img_id)
        # Inference on absolute image path
        try:
            boxes = det.detect(rec['image_path'])
        except Exception as e:
            print(f'[{detector_alias}] skip {rec["image_path"]}: {e}')
            continue
        for b in boxes:
            preds.append({
                'image_id': img_id,
                'bbox': [float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])],
                'score': float(b[4]) if len(b) >= 5 else 1.0,
                'category_id': 1,
            })
        done += 1
        if done % 50 == 0:
            print(f'  [{detector_alias}] {done}/{n} images, {len(preds)} preds so far, {time.time()-t0:.1f}s')
    print(f'[{detector_alias}] DONE: {done} images, {len(preds)} preds in {time.time()-t0:.1f}s')
    return preds

## 4. Run evaluation per detector and save to Drive

In [ ]:
DETECTORS = cfg['detectors']
CONF_THR = cfg['confidence_threshold']

all_summaries = []
for alias in DETECTORS:
    print(f'\n=== {alias} ===')
    preds = run_detector_on_inria(alias, gt_records, conf_thr=0.001)
    # COCOeval with the same confidence filter the config specifies
    result = evaluate_map(preds, gt_records, confidence_threshold=CONF_THR, category_id=1)
    summary = {
        'detector': alias,
        'mAP_50_95': result.map_5095,
        'AP50': result.ap50,
        'AP75': result.ap75,
        'num_images': result.num_images,
        'num_gt_total': result.num_gt_total,
        'num_pred_total': result.num_pred_total,
        'num_pred_after_conf': result.num_pred_after_conf,
        'confidence_threshold': result.confidence_threshold,
        'note': result.note,
    }
    print(json.dumps(summary, indent=2))
    # Save both locally and to Drive
    local_path = Path(cfg['output_dir']) / f'summary_{alias}.json'
    local_path.parent.mkdir(parents=True, exist_ok=True)
    write_map_result_json(result, local_path)
    shutil.copy(local_path, DRIVE_RESULT_DIR / f'summary_{alias}.json')
    all_summaries.append(summary)

with open(DRIVE_RESULT_DIR / 'all_summaries.json', 'w') as f:
    json.dump(all_summaries, f, indent=2)
print('\nAll summaries uploaded to:', DRIVE_RESULT_DIR)

## 5. Result summary table

In [ ]:
import pandas as pd
df = pd.DataFrame(all_summaries)
df = df[['detector', 'mAP_50_95', 'AP50', 'AP75', 'num_images', 'num_gt_total', 'num_pred_total', 'num_pred_after_conf']]
print(df.to_string(index=False))
df.to_csv(DRIVE_RESULT_DIR / 'summary_table.csv', index=False)

## 6. Reviewer prompt

After mAP numbers are produced, write a reviewer pass to
`docs/reviews/review-s1-quant.md` covering:

- Same-tier comparison validity (no cross-tier claims).
- Calibration check: num_pred_after_conf << num_pred_total.
- Anti-overclaim: do NOT call Nano results 'best' — they are anchor only.
- Next step: which tier (M/L) and which detector should advance to Phase 9?